In [1]:
# pip install pandas
# pip install nltk

In [2]:
# pip install transformers datasets scikit-learn torch

In [3]:
import numpy as np
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
nltk.download("stopwords")
pd.set_option("display.max_rows",None)
pd.set_option("display.max_columns",None)

[nltk_data] Downloading package stopwords to /home/md-ismail-
[nltk_data]     quraishi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
path = "data/train.csv"
df= pd.read_csv(path)

In [5]:
df.head(2)

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0


In [6]:
df.iloc[7]["prompt"].replace("\\n"," ")

'["\\"Bacteria is life on Mars but a heartbeat isn\'t life on earth?\\" What is this quote from?"]'

In [7]:
x=df["prompt"][7].replace("["," ").replace("]"," ").replace('"'," ").replace("\\"," ").replace("?"," ").strip()

In [8]:
x=" ".join(x.split())
" ".join([i if i not in stopwords.words("english") else "" for i in x.split()]).strip()

'Bacteria  life  Mars   heartbeat  life  earth What   quote'

In [9]:
print(f"shape: {df.shape}")
print("-"*100)
print(f"columns: {df.columns}")
print("-"*100)
print(df.head(3))

shape: (57477, 9)
----------------------------------------------------------------------------------------------------
columns: Index(['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b',
       'winner_model_a', 'winner_model_b', 'winner_tie'],
      dtype='object')
----------------------------------------------------------------------------------------------------
      id             model_a         model_b  \
0  30192  gpt-4-1106-preview      gpt-4-0613   
1  53567           koala-13b      gpt-4-0613   
2  65089  gpt-3.5-turbo-0613  mistral-medium   

                                              prompt  \
0  ["Is it morally right to try to have a certain...   
1  ["What is the difference between marriage lice...   
2  ["explain function calling. how would you call...   

                                          response_a  \
0  ["The question of whether it is morally right ...   
1  ["A marriage license is a legal document that ...   
2  ["Function calling is the pro

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57477 entries, 0 to 57476
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              57477 non-null  int64 
 1   model_a         57477 non-null  object
 2   model_b         57477 non-null  object
 3   prompt          57477 non-null  object
 4   response_a      57477 non-null  object
 5   response_b      57477 non-null  object
 6   winner_model_a  57477 non-null  int64 
 7   winner_model_b  57477 non-null  int64 
 8   winner_tie      57477 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 3.9+ MB


In [11]:
df.describe()

,id,winner_model_a,winner_model_b,winner_tie
count,5.747700e+04,57477.000000,57477.000000,57477.000000
mean,2.142564e+09,0.349079,0.341911,0.309011
std,1.238327e+09,0.476683,0.474354,0.462090
min,3.019200e+04,0.000000,0.000000,0.000000
25%,1.071821e+09,0.000000,0.000000,0.000000
50%,2.133658e+09,0.000000,0.000000,0.000000
75%,3.211645e+09,1.000000,1.000000,1.000000
max,4.294947e+09,1.000000,1.000000,1.000000


In [12]:
def get_count_colwise(df,col):
    x= df[col].value_counts().reset_index()
    x.index= x[col]
    x.drop(col, axis=1, inplace=True)
    return x

model_a_vc= get_count_colwise(df.copy(), "model_a")
model_b_vc= get_count_colwise(df.copy(), "model_b")
winner_a_count= get_count_colwise(df.copy(), "winner_model_a")
winner_b_count= get_count_colwise(df.copy(), "winner_model_b")
winner_tie_count= get_count_colwise(df.copy(), "winner_tie")

In [13]:
model_a_vc.T

model_a,gpt-4-1106-preview,gpt-3.5-turbo-0613,gpt-4-0613,claude-2.1,gpt-4-0314,claude-instant-1,claude-1,vicuna-33b,mixtral-8x7b-instruct-v0.1,mistral-medium,vicuna-13b,gpt-3.5-turbo-1106,llama-2-70b-chat,llama-2-13b-chat,claude-2.0,zephyr-7b-beta,palm-2,llama-2-7b-chat,openchat-3.5,mistral-7b-instruct,wizardlm-70b,wizardlm-13b,koala-13b,vicuna-7b,oasst-pythia-12b,codellama-34b-instruct,gemini-pro-dev-api,gemini-pro,pplx-70b-online,alpaca-13b,yi-34b-chat,gpt-3.5-turbo-0314,chatglm-6b,pplx-7b-online,RWKV-4-Raven-14B,tulu-2-dpo-70b,gpt-4-0125-preview,starling-lm-7b-alpha,qwen-14b-chat,chatglm3-6b,stripedhyena-nous-7b,fastchat-t5-3b,openhermes-2.5-mistral-7b,mpt-7b-chat,solar-10.7b-instruct-v1.0,deepseek-llm-67b-chat,gpt-3.5-turbo-0125,dolly-v2-12b,stablelm-tuned-alpha-7b,guanaco-33b,llama2-70b-steerlm-chat,mpt-30b-chat,chatglm2-6b,qwen1.5-72b-chat,llama-13b,gpt4all-13b-snoozy,zephyr-7b-alpha,dolphin-2.2.1-mistral-7b,nous-hermes-2-mixtral-8x7b-dpo,falcon-180b-chat,openchat-3.5-0106,qwen1.5-7b-chat,qwen1.5-4b-chat,mistral-7b-instruct-v0.2
count,3678,3553,3099,2859,2087,2085,1955,1843,1741,1706,1705,1686,1675,1276,1272,1207,1019,856,823,804,800,798,795,765,753,747,729,719,714,709,696,646,612,601,588,587,567,561,539,503,491,487,474,451,434,430,417,397,394,347,346,311,283,278,278,216,210,199,163,145,108,106,100,54


In [14]:
model_b_vc.T

model_b,gpt-4-1106-preview,gpt-3.5-turbo-0613,gpt-4-0613,claude-2.1,claude-instant-1,gpt-4-0314,claude-1,vicuna-33b,mixtral-8x7b-instruct-v0.1,llama-2-70b-chat,vicuna-13b,gpt-3.5-turbo-1106,mistral-medium,llama-2-13b-chat,zephyr-7b-beta,claude-2.0,palm-2,llama-2-7b-chat,wizardlm-70b,vicuna-7b,mistral-7b-instruct,openchat-3.5,koala-13b,wizardlm-13b,gemini-pro-dev-api,yi-34b-chat,oasst-pythia-12b,codellama-34b-instruct,gemini-pro,pplx-70b-online,alpaca-13b,gpt-3.5-turbo-0314,pplx-7b-online,chatglm-6b,tulu-2-dpo-70b,gpt-4-0125-preview,starling-lm-7b-alpha,RWKV-4-Raven-14B,fastchat-t5-3b,qwen-14b-chat,chatglm3-6b,openhermes-2.5-mistral-7b,mpt-7b-chat,solar-10.7b-instruct-v1.0,gpt-3.5-turbo-0125,stripedhyena-nous-7b,dolly-v2-12b,stablelm-tuned-alpha-7b,deepseek-llm-67b-chat,guanaco-33b,llama2-70b-steerlm-chat,mpt-30b-chat,chatglm2-6b,qwen1.5-72b-chat,llama-13b,zephyr-7b-alpha,gpt4all-13b-snoozy,dolphin-2.2.1-mistral-7b,nous-hermes-2-mixtral-8x7b-dpo,falcon-180b-chat,openchat-3.5-0106,qwen1.5-7b-chat,qwen1.5-4b-chat,mistral-7b-instruct-v0.2
count,3709,3530,3066,2724,2051,2035,2023,1877,1804,1753,1743,1666,1609,1331,1194,1184,958,937,844,826,813,809,803,782,757,751,741,727,719,706,694,656,649,649,613,593,573,570,534,533,486,478,477,465,444,423,403,377,365,337,321,287,281,273,269,202,192,174,162,141,136,102,100,46


In [15]:
print("winner_a count:")
print(winner_a_count)
print("-"*30)
print("winner_b count:")
print(winner_b_count)
print("-"*30)
print("winner_tie count:")
print(winner_tie_count)

winner_a count:
                count
winner_model_a       
0               37413
1               20064
------------------------------
winner_b count:
                count
winner_model_b       
0               37825
1               19652
------------------------------
winner_tie count:
            count
winner_tie       
0           39716
1           17761


In [16]:
stopwords_= stopwords.words("english")
punc_= string.punctuation
def target_encoder(target):
    a=target.iloc[0]
    b= target.iloc[1]
    tie= target.iloc[2]
    
    if a==1:
        return 1
    elif b==1:
        return 2
    elif tie==1:
        return 0

def get_clean_text(sentence):
    if isinstance(sentence, str):
        sentence= "".join([char if char not in punc_ else "" for char in sentence]).strip()
        sentence= "".join([t if t not in "0123456789" else "" for t in sentence])
        return sentence.lower()

target_cols= ["winner_model_a", "winner_model_b", "winner_tie"]
df["target"]= df[target_cols].apply(target_encoder, axis=1)
# df.drop(target_cols, axis=1, inplace=True)

In [18]:
df["prompt_cleaned"]=df["prompt"].apply(get_clean_text)
df["response_a_cleaned"]=df["response_a"].apply(get_clean_text)
df["response_b_cleaned"]=df["response_b"].apply(get_clean_text)

df["prompt_cleaned"]=df["prompt_cleaned"].apply(lambda sentence: " ".join([text if text not in stopwords_ else "" for text in sentence.split()]))
df["response_a_cleaned"]=df["response_a_cleaned"].apply(lambda sentence: " ".join([text if text not in stopwords_ else "" for text in sentence.split()]))
df["response_b_cleaned"]=df["response_b_cleaned"].apply(lambda sentence: " ".join([text if text not in stopwords_ else "" for text in sentence.split()]))

# df.drop(["prompt","response_a", "response_b"], axis=1, inplace=True)

In [19]:
df.head(2)

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,target,prompt_cleaned,response_a_cleaned,response_b_cleaned
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0,1,morally right try certain percentage fe...,question whether morally right aim cert...,ai dont personal beliefs opinions however...
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0,2,difference marriage license marriage cert...,marriage license legal document allows co...,marriage license marriage certificate two ...


In [20]:
df2= pd.DataFrame()
df2["id"]= df["id"]
df2["text_combined"]= df["prompt_cleaned"]+" "+df["response_a_cleaned"]+" "+df['response_b_cleaned']
df2["target"]= df["target"]
df2.head(3)

,id,text_combined,target
0,30192,morally right try certain percentage fe...,1
1,53567,difference marriage license marriage cert...,2
2,65089,explain function calling would call functio...,0


In [21]:
# nltk.download("wordnet")
lemmatizer= WordNetLemmatizer()
df2["text_combined"]= df2["text_combined"].apply(lambda x: " ".join([lemmatizer.lemmatize(text) for text in x.split()]))
tfidf= TfidfVectorizer(ngram_range=(1,3))

In [22]:
x=tfidf.fit_transform(df2["text_combined"])

In [25]:
y=df2["target"]

In [26]:
X_train, X_test, y_train, y_test=train_test_split(x, y, random_state=1234, test_size=0.15)

In [27]:
from sklearn.naive_bayes import MultinomialNB
model= MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [28]:
pred= model.predict(x)
pred_train= model.predict(X_train)
pred_test= model.predict(X_test)

In [29]:
df["prediction"]= pred

In [30]:
print(classification_report(y_train, pred_train))

              precision    recall  f1-score   support

           0       0.94      0.63      0.75     15079
           1       0.80      0.98      0.88     17034
           2       0.91      0.97      0.94     16742

    accuracy                           0.87     48855
   macro avg       0.88      0.86      0.86     48855
weighted avg       0.88      0.87      0.86     48855



In [31]:
print(classification_report(y_test, pred_test))

              precision    recall  f1-score   support

           0       0.53      0.14      0.23      2682
           1       0.36      0.66      0.47      3030
           2       0.34      0.28      0.31      2910

    accuracy                           0.37      8622
   macro avg       0.41      0.36      0.33      8622
weighted avg       0.41      0.37      0.34      8622

